# Generating Text With A PretrainedLLM

In [1]:
import torch

print("Pytorch Version: ", torch.__version__)

Pytorch Version:  2.11.0+cu128


In [2]:
if torch.cuda.is_available():
    print(f"Cuda/ROCm GPU: {torch.cuda.get_device_name(0)}")
else:
    print("The device is running on cpu.!")

Cuda/ROCm GPU: Tesla T4


In [3]:
%pip install git+https://github.com/rasbt/reasoning-from-scratch.git


  Cloning https://github.com/rasbt/reasoning-from-scratch.git to /tmp/pip-req-build-jz9puda7
  Running command git clone --filter=blob:none --quiet https://github.com/rasbt/reasoning-from-scratch.git /tmp/pip-req-build-jz9puda7
  Resolved https://github.com/rasbt/reasoning-from-scratch.git to commit 3c1e94671ea31a8b93f6deb92d55dcbc7972bf3b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 76.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

## Preparing input text to for LLM

In [4]:
from reasoning_from_scratch.qwen3 import download_qwen3_small

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="Qwen3")


In [5]:
# Load the tokenizer

from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer

tokenizer_path = Path("Qwen3") / "tokenizer-base.json"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

tokenizer_path


PosixPath('Qwen3/tokenizer-base.json')

In [6]:
tokenizer

In [7]:
# see it is working perfectly or not

prompt = "Explain the large language model"
input_token_ids_list = tokenizer.encode(prompt)
input_token_ids_list

[840, 20772, 279, 3460, 4128, 1614]

In [8]:
text = tokenizer.decode(input_token_ids_list)
text

'Explain the large language model'

In [9]:
for i in input_token_ids_list:
    print(f"{i} -----> {tokenizer.decode([i])}")

840 -----> Ex
20772 -----> plain
279 ----->  the
3460 ----->  large
4128 ----->  language
1614 ----->  model


### Get the device automatically

In [10]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Device is using GPU")
        return device
    else:
        device=torch.device("cpu")
        print("Device is using CPU")
        return device

In [11]:
# to download model weight as well we can do with tokenizer_only=False

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [12]:
device = get_device()

Device is using GPU


In [13]:
from reasoning_from_scratch.qwen3 import QWEN_CONFIG_06_B, Qwen3Model

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B) # Instantiate a Qwen3 model with random weight as placeholders

model.load_state_dict(torch.load(model_path)) # Load the pretrained weights into the model
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [14]:
# lets see text generation in each iteration of the autoregressive process

prompt = "Explain large language models."

input_token_ids_list = tokenizer.encode(prompt)
print(f"Number of input tokens: {len(input_token_ids_list)}")

input_tensor = torch.tensor(input_token_ids_list) # convert python list into python sensor
input_tensor_fmt = input_tensor.unsqueeze(0) # Added an aditional dimension
input_tensor_fmt = input_tensor_fmt.to(device)

with torch.inference_mode():
    output_tensor = model(input_tensor_fmt) # generate the output

output_tensor_fmt = output_tensor.squeeze(0) # Removed the extra dimension
print("Formatted Output tensor shape: ", output_tensor_fmt.shape)

Number of input tokens: 6
Formatted Output tensor shape:  torch.Size([6, 151936])


In [15]:
# find the last position, because it predicts the next unseen token
last_token = output_tensor_fmt[-1]
print(last_token)

tensor([ 7.3125,  1.9375,  7.9688,  ..., -2.4844, -2.4844, -2.4844],
       device='cuda:0', dtype=torch.bfloat16)


In [16]:
# we will use argmax to find the position of the largest value score in the tensor

print(torch.argmax(last_token, dim=-1, keepdim=True))

tensor([20286], device='cuda:0')


In [17]:
# Lets see what is the text value

print(tokenizer.decode([20286]))

 Large


### Coding a minimal text generation function

Lets create a simple full text generation function

In [18]:
@torch.inference_mode() # it disable gradient tracking for speed and memory efficiently
def generate_text_basic_stream( 
        model,
        token_ids,
        max_new_tokens,
        eos_token_id = None
    ):
    model.eval() # switch model to evaluation mode to enable deterministic behavior(best practice)

    for _ in range(max_new_tokens):
        out = model(token_ids)[:, -1] # get the score of last token
        next_token = torch.argmax(out, dim=-1, keepdim=True)

        if(eos_token_id is not None and torch.all(next_token == eos_token_id)): # stop all the sequences in the batch have generated EOS
            break

        yield next_token # yield each token as soon as its generated

        token_ids = torch.cat([token_ids, next_token], dim=1) # append the newly predicted token to the sequence



In [19]:
# lets test the function for 100 max tokens

prompt = "Explain large language models in a single sentence"

input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device = device
).unsqueeze(0)
max_input_tokens =100

for token in generate_text_basic_stream(model = model, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens):
    token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
    print(tokenizer.decode(token_id), 
        end="",
        flush=True) # deactivates buffering so tokens are printed live

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.<|endoftext|>Human language is a complex and dynamic system that requires sophisticated models to understand and generate meaningful responses. Large language models are designed to mimic this complexity by using vast amounts of data and advanced algorithms to learn patterns and relationships in language. These models can be trained on a wide range of sources, including text, speech, and images

In [20]:
print(tokenizer.encode("<|endoftext|>"))

[151643]


In [ ]:
# This token id(151643) is also saved via the tokenizer.eos_token_id



for token in generate_text_basic_stream(model = model, token_ids=input_token_ids_tensor, max_new_tokens=max_input_tokens, eos_token_id = tokenizer.eos_token_id):
    token_id = token.squeeze(0).tolist() # convert output token IDs from pytorch tensor to Python list)
    print(tokenizer.decode(token_id), 
        end="",
        flush=True) # deactivates buffering so tokens are printed live

# here will text generation stop when end_of_sequence apprear

.
Large language models are artificial intelligence systems that can understand, generate, and process human language, enabling them to perform tasks such as translation, summarization, and question answering.